In [128]:
# проверяю, что мы, действ, на сервере
!ls

data  latest_shape.png	outputs  requirements.txt  sc.bash  stablediffusion


In [129]:
!pip install -r requirements.txt
# For CUDA (becauase we have NVIDIA on the server)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu118


In [130]:
!pip install ipywidgets
# !jupyter nbextension enable --py widgetsnbextension
# !jupyter nbextension enable --py --sys-prefix widgetsnbextension

Defaulting to user installation because normal site-packages is not writeable


In [131]:
import os
import torch
import numpy as np
from pathlib import Path
from typing import Tuple, List, Dict
import json
import cv2
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt

import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset, ConcatDataset, Subset
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, DiffusionPipeline
from diffusers import DDPMScheduler, DDIMScheduler
from PIL import ImageDraw, ImageOps
import torchvision.models as models

from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import warnings
warnings.filterwarnings('ignore')


# CONFIGURATION

In [ ]:
class Config:
    """Глобальная конфигурация"""
    # Paths
    DATA_DIR = Path("./data")
    OUTPUT_DIR = Path("./outputs")
    SYNTHETIC_DIR = OUTPUT_DIR / "synthetic_images"
    CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
    RESULTS_DIR = OUTPUT_DIR / "results"

    # Dataset
    DATASET = "CIFAR10"  # CIFAR10 или STL10
    NUM_CLASSES = 10
    IMBALANCE_RATIO = 0.1  # Часть, которую удалим у некоторого класса, чтобы сделать его редким
    RARE_CLASS = 3  # 'cat' в CIFAR10
    NUM_SYNTHETIC = 50  # Количество генерируемых синтетических изображений

    # Model
    MODEL_TYPE = "ResNet50"  # ResNet50 или ViT
    BATCH_SIZE = 64
    NUM_EPOCHS = 100
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4

    # Generation
    USE_CONTROLNET = True
    CONTROLNET_TYPE = "canny"  # canny, openpose, depth, segmentation
    STABLE_DIFFUSION_MODEL = "runwayml/stable-diffusion-v1-5"
    CONTROLNET_MODEL = "lllyasviel/sd-controlnet-canny"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED = 134

    # Generation params
    GUIDANCE_SCALE = 8.0
    NUM_INFERENCE_STEPS = 52
    NUM_IMAGES_PER_PROMPT = 1

    # Class prompts for CIFAR10
    CLASS_PROMPTS = {
        0: "a clear photo of an airplane",
        1: "a clear photo of a car",
        2: "a clear photo of a bird",
        3: "detailed photo of a cat",
        4: "a clear photo of a deer",
        5: "a clear photo of a dog",
        6: "a clear photo of a frog",
        7: "a clear photo of a horse",
        8: "a clear photo of a ship",
        9: "a clear photo of a truck",
    }

    def __init__(self):
        # Создаем директории
        self.DATA_DIR.mkdir(exist_ok=True)
        self.OUTPUT_DIR.mkdir(exist_ok=True)
        self.SYNTHETIC_DIR.mkdir(exist_ok=True)
        self.CHECKPOINT_DIR.mkdir(exist_ok=True)
        self.RESULTS_DIR.mkdir(exist_ok=True)


config = Config()
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)


# DATA LOADING AND PREPARATION

In [133]:
class ImbalancedCIFAR10Dataset(Dataset):
    """Датасет CIFAR10 с имитацией дисбаланса классов"""

    def __init__(self, train=True, transform=None, imbalance_ratio=1.0, rare_class=None):
        self.cifar10 = datasets.CIFAR10(
            root=config.DATA_DIR,
            train=train,
            download=True,
            transform=transform
        )

        self.data_indices = self._create_imbalanced_indices(imbalance_ratio, rare_class)
        self.transform = transform

    def _create_imbalanced_indices(self, imbalance_ratio, rare_class):
        """Создаем индексы для имитации дисбаланса"""
        indices = []
        class_counts = {}

        # Группируем индексы по классам
        for idx, (_, label) in enumerate(self.cifar10):
            if label not in class_counts:
                class_counts[label] = []
            class_counts[label].append(idx)

        # Для редкого класса берем только часть
        if rare_class is not None:
            num_rare = int(len(class_counts[rare_class]) * imbalance_ratio)
            class_counts[rare_class] = class_counts[rare_class][:num_rare]

        # Собираем все индексы
        for class_idx, idx_list in class_counts.items():
            indices.extend(idx_list)

        return indices

    def __len__(self):
        return len(self.data_indices)

    def __getitem__(self, idx):
        actual_idx = self.data_indices[idx]
        image, label = self.cifar10[actual_idx]
        return image, label


def get_data_loaders():
    """Загружаем датасеты"""

    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2023, 0.1994, 0.2010))
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2023, 0.1994, 0.2010))
    ])

    # Полный датасет для обучения
    train_full = ImbalancedCIFAR10Dataset(
        train=True,
        transform=transform_train,
        imbalance_ratio=1.0,
        rare_class=None
    )

    # Имбалансированный датасет
    train_imbalanced = ImbalancedCIFAR10Dataset(
        train=True,
        transform=transform_train,
        imbalance_ratio=config.IMBALANCE_RATIO,
        rare_class=config.RARE_CLASS
    )

    # Тестовый датасет
    test_dataset = datasets.CIFAR10(
        root=config.DATA_DIR,
        train=False,
        download=True,
        transform=transform_test
    )

    train_loader_full = DataLoader(
        train_full, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=32
    )

    train_loader_imbalanced = DataLoader(
        train_imbalanced, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=32
    )

    test_loader = DataLoader(
        test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=32
    )

    return {
        'train_full': train_loader_full,
        'train_imbalanced': train_imbalanced,
        'test': test_loader,
        'train_full_dataset': train_full
    }

# IMAGE GENERATION WITH STABLE DIFFUSION

## Get random shape for control image edge map

In [134]:
def get_random_shape(size: Tuple[int, int] = (512, 512)) -> Image.Image:
    img = Image.new('RGB', size, color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

     # Случайные координаты для фигуры
    margin = size[0] // 8  # отступ от краев
    x_min, y_min = margin, margin
    x_max, y_max = size[0] - margin, size[1] - margin
    
    shape_type = np.random.choice(['rect', 'pentagon'])
    
    if shape_type == 'rect':
        # Случайный прямоугольник
        x1 = np.random.randint(x_min, x_max - 100)
        y1 = np.random.randint(y_min, y_max - 100)
        w = np.random.randint(150, 300)
        h = np.random.randint(150, 300)
        x2, y2 = x1 + w, y1 + h
        
        draw.rectangle([x1, y1, x2, y2], fill=(0, 0, 0), outline=(0, 0, 0), width=3)
        
    else:
        # Случайный пятиугольник
        center_x = np.random.randint(x_min + 100, x_max - 100)
        center_y = np.random.randint(y_min + 100, y_max - 100)
        radius = np.random.randint(100, 180)
        
        # Генерируем 5 случайных углов
        angles = np.random.uniform(0, 2*np.pi, 5)
        angles.sort()  # для правильного порядка
        
        points = []
        for angle in angles:
            x = center_x + radius * np.cos(angle)
            y = center_y + radius * np.sin(angle)
            points.append((x, y))
        
        draw.polygon(points, fill=(0, 0, 0), outline=(0, 0, 0), width=3)
        
    save_path = "latest_shape.png"
    img.save(save_path)
    
    return img

## Synth data generator

In [135]:
class SyntheticDataGenerator:
    """Генератор синтетических изображений используя Stable Diffusion + ControlNet"""

    def __init__(self, class_idx: int, class_name: str):
        self.class_idx = class_idx
        self.class_name = class_name
        self.device = config.DEVICE
        self.pipe = None
        self._init_pipeline()

    def _init_pipeline(self):
        """Инициализируем пайплайн Stable Diffusion"""
        print(f"[Gen] Загружаем Stable Diffusion для класса '{self.class_name}'...")

        # # NOTE: fix tqdm UNUSED
        # import os
        # os.environ['TQDM_DISABLE'] = '0'  # Включаем tqdm
        # os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'  # Отключаем HF progress bars

        if config.USE_CONTROLNET:
            try:
                controlnet = ControlNetModel.from_pretrained(
                    config.CONTROLNET_MODEL,
                    torch_dtype=torch.float16,
                    use_safetensors=True
                    # tqdm_callback=lambda *args, **kwargs: None # NOTE: fix tqdm
                )

                self.pipe = StableDiffusionControlNetPipeline.from_pretrained(
                    config.STABLE_DIFFUSION_MODEL,
                    controlnet=controlnet,
                    torch_dtype=torch.float16,
                    use_safetensors=True
                    # tqdm_callback=lambda *args, **kwargs: None # NOTE: fix tqdm
                )
            except Exception as e:
                print(f"[Gen] ControlNet не загрузился ({e}), используем обычный Stable Diffusion")
                self.pipe = DiffusionPipeline.from_pretrained(
                    config.STABLE_DIFFUSION_MODEL,
                    torch_dtype=torch.float16,
                    use_safetensors=True
                    # tqdm_callback=lambda *args, **kwargs: None # NOTE: fix tqdm
                )
        else:
            self.pipe = DiffusionPipeline.from_pretrained(
                config.STABLE_DIFFUSION_MODEL,
                torch_dtype=torch.float16,
                use_safetensors=True
            )

        self.pipe.to(self.device)
        self.pipe.enable_attention_slicing()

        # Для экономии памяти
        if hasattr(self.pipe, 'enable_sequential_cpu_offload'):
            self.pipe.enable_sequential_cpu_offload()

    def _generate_control_image(self, size: Tuple[int, int] = (512, 512)) -> Image.Image:
        """Генерируем контрольное изображение (edge map для Canny)"""
        if config.CONTROLNET_TYPE == "canny":
            # Генерируем случайное изображение краев
            control_img = Image.new('RGB', size, color='white')
            draw = ImageDraw.Draw(control_img)

            # Рисуем случайные линии для имитации краев
            for _ in range(np.random.randint(3, 8)):
                x1 = np.random.randint(0, size[0])
                y1 = np.random.randint(0, size[1])
                x2 = np.random.randint(0, size[0])
                y2 = np.random.randint(0, size[1])
                draw.line([(x1, y1), (x2, y2)], fill='black', width=2)

            # return control_img
            return get_random_shape(size)
        else:
            # Для других типов возвращаем нейтральное изображение
            return Image.new('RGB', size, color='gray')

    def generate_images(self, num_images: int = 10) -> List[Image.Image]:
        """Генерируем синтетические изображения"""
        print(f"[Gen] Генерируем {num_images} изображений для класса '{self.class_name}'...")

        prompt = config.CLASS_PROMPTS.get(self.class_idx, f"a photo of a {self.class_name}")
        negative_prompt = "blurry, low quality, distorted"

        generated_images = []

        output_dir = config.SYNTHETIC_DIR / self.class_name
        output_dir.mkdir(parents=True, exist_ok=True)

        for i in tqdm(range(num_images), desc=f"Generating {self.class_name}"):
            try:
                if config.USE_CONTROLNET and isinstance(self.pipe, StableDiffusionControlNetPipeline):
                    # control_image = self._generate_control_image((512, 512))
                    control_image = Image.new('RGB', (512, 512), color='black')

                    image = self.pipe(
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        image=control_image,
                        num_inference_steps=config.NUM_INFERENCE_STEPS,
                        guidance_scale=config.GUIDANCE_SCALE,
                        height=256,
                        width=256,
                    ).images[0]
                else:
                    image = self.pipe(
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        num_inference_steps=config.NUM_INFERENCE_STEPS,
                        guidance_scale=config.GUIDANCE_SCALE,
                        height=256,
                        width=256,
                    ).images[0]

                original_path = output_dir / f"{self.class_name}_orig_{i:04d}.png"
                image.save(original_path)

                # Преобразуем в CIFAR10 размер (32x32)
                image = image.resize((32, 32), Image.Resampling.LANCZOS)
                generated_images.append(image)

            except Exception as e:
                print(f"[Gen] Ошибка при генерации: {e}")
                continue

        return generated_images

    def save_images(self, images: List[Image.Image], output_dir: Path):
        """Сохраняем сгенерированные изображения"""
        output_dir.mkdir(parents=True, exist_ok=True)

        for i, img in enumerate(images):
            save_path = output_dir / f"{self.class_name}_{i:04d}.png"
            img.save(save_path)

        print(f"[Gen] Сохранено {len(images)} изображений в {output_dir}")


def generate_synthetic_dataset():
    """Основной цикл генерации синтетического датасета"""
    generator = SyntheticDataGenerator(config.RARE_CLASS, "cat")

    # Генерируем изображения
    synthetic_images = generator.generate_images(config.NUM_SYNTHETIC)

    # Сохраняем
    generator.save_images(synthetic_images, config.SYNTHETIC_DIR)

    return synthetic_images

# MODEL

In [ ]:
def create_model(model_type: str = "ResNet50") -> nn.Module:
    """Создаем модель для классификации"""

    if model_type == "ResNet50":
        model = models.resnet50(pretrained=False)
        model.fc = nn.Linear(model.fc.in_features, config.NUM_CLASSES)

    elif model_type == "ResNet18":
        model = models.resnet18(pretrained=False)
        model.fc = nn.Linear(model.fc.in_features, config.NUM_CLASSES)

    elif model_type == "ViT":
        # Vision Transformer - требует timm library
        try:
            import timm
            model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=config.NUM_CLASSES)
        except ImportError:
            print("[Model] timm не установлен, используем ResNet50 вместо ViT")
            model = models.resnet50(pretrained=False)
            model.fc = nn.Linear(model.fc.in_features, config.NUM_CLASSES)

    else:
        raise ValueError(f"Unknown model type: {model_type}")

    return model

# TRAINING

In [137]:
class Trainer:
    """Тренер для обучения моделей"""

    def __init__(self, model: nn.Module, device: torch.device):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        self.scheduler = CosineAnnealingLR(
            self.optimizer,
            T_max=config.NUM_EPOCHS,
            eta_min=1e-6
        )
        self.history = {'train_loss': [], 'val_acc': [], 'val_balanced_acc': []}

    def train_epoch(self, train_loader: DataLoader) -> float:
        """Тренируем одну эпоху"""
        self.model.train()
        total_loss = 0

        for images, labels in tqdm(train_loader, desc="Training"):
            images, labels = images.to(self.device), labels.to(self.device)

            self.optimizer.zero_grad()

            outputs = self.model(images)
            loss = self.criterion(outputs, labels)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        self.history['train_loss'].append(avg_loss)
        return avg_loss

    def evaluate(self, test_loader: DataLoader) -> Tuple[float, float]:
        """Оцениваем модель"""
        self.model.eval()
        correct = 0
        total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc="Evaluating"):
                images, labels = images.to(self.device), labels.to(self.device)

                outputs = self.model(images)
                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        accuracy = 100 * correct / total
        balanced_acc = 100 * balanced_accuracy_score(all_labels, all_preds)

        self.history['val_acc'].append(accuracy)
        self.history['val_balanced_acc'].append(balanced_acc)

        return accuracy, balanced_acc

    def train(self, train_loader: DataLoader, test_loader: DataLoader, num_epochs: int):
        """Полный цикл обучения"""
        print(f"[Train] Начинаем обучение на {num_epochs} эпох...")

        for epoch in range(num_epochs):
            train_loss = self.train_epoch(train_loader)
            val_acc, val_balanced_acc = self.evaluate(test_loader)

            self.scheduler.step()

            if (epoch + 1) % 10 == 0:
                print(f"[Epoch {epoch+1}/{num_epochs}] "
                      f"Loss: {train_loss:.4f} | "
                      f"Acc: {val_acc:.2f}% | "
                      f"Balanced Acc: {val_balanced_acc:.2f}%")

        return self.history


# AUGMENTED DATASET WITH SYNTHETIC IMAGES

In [138]:
class AugmentedCIFAR10Dataset(Dataset):
    """Датасет с добавленными синтетическими изображениями"""

    def __init__(self, base_dataset, synthetic_dir: Path, synthetic_class: int, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform
        self.synthetic_class = synthetic_class
        self.synthetic_images = []
        self.synthetic_labels = []

        # Загружаем синтетические изображения
        self._load_synthetic_images(synthetic_dir)

    def _load_synthetic_images(self, synthetic_dir: Path):
        """Загружаем сгенерированные изображения"""
        if not synthetic_dir.exists():
            print(f"[Dataset] Синтетическая директория не найдена: {synthetic_dir}")
            return

        for img_path in synthetic_dir.glob("*.png"):
            try:
                img = Image.open(img_path).convert('RGB')
                self.synthetic_images.append(img)
                self.synthetic_labels.append(self.synthetic_class)
            except Exception as e:
                print(f"[Dataset] Ошибка загрузки {img_path}: {e}")

        print(f"[Dataset] Загружено {len(self.synthetic_images)} синтетических изображений")

    def __len__(self):
        return len(self.base_dataset) + len(self.synthetic_images)

    def __getitem__(self, idx):
        if idx < len(self.base_dataset):
            image, label = self.base_dataset[idx]
        else:
            synthetic_idx = idx - len(self.base_dataset)
            image = self.synthetic_images[synthetic_idx]
            label = self.synthetic_labels[synthetic_idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [139]:
def run_complete_pipeline():
    """Запускаем полный пайплайн"""

    print("=" * 80)
    print("ЗАДАНИЕ 2.5: СИНТЕТИЧЕСКИЕ ДАННЫЕ ЧЕРЕЗ STABLE DIFFUSION + CONTROLNET")
    print("=" * 80)

    # Загружаем данные
    print("\n[1] Загружаем датасеты...")
    data_loaders = get_data_loaders()

    # Генерируем синтетические данные (опционально, требует GPU памяти)
    print("\n[2] Генерируем синтетические изображения...")
    try:
        generate_synthetic_dataset()
        print("[OK] Синтетические данные готовы!")
    except Exception as e:
        print(f"[WARNING] Не удалось сгенерировать синтетику: {e}")
        print("[INFO] Продолжаем с предварительно сгенерированными изображениями")

    # Подготавливаем трансформации
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2023, 0.1994, 0.2010))
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2023, 0.1994, 0.2010))
    ])

    # Обучение: Сценарий 1 - Без синтетики
    print("\n[3] Сценарий 1: Обучение БЕЗ синтетических данных...")
    model1 = create_model(config.MODEL_TYPE)
    trainer1 = Trainer(model1, config.DEVICE)

    train_loader_imbalanced = DataLoader(
        data_loaders['train_imbalanced'],
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=32
    )

    history1 = trainer1.train(
        train_loader_imbalanced,
        data_loaders['test'],
        config.NUM_EPOCHS
    )

    # Обучение: Сценарий 2 - С синтетикой
    print("\n[4] Сценарий 2: Обучение С синтетическими данными...")

    # Создаем аугментированный датасет
    augmented_dataset = AugmentedCIFAR10Dataset(
        data_loaders['train_imbalanced'],
        config.SYNTHETIC_DIR,
        config.RARE_CLASS,
        transform=transform_train
    )

    train_loader_augmented = DataLoader(
        augmented_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=32
    )

    model2 = create_model(config.MODEL_TYPE)
    trainer2 = Trainer(model2, config.DEVICE)

    history2 = trainer2.train(
        train_loader_augmented,
        data_loaders['test'],
        config.NUM_EPOCHS
    )

    # Результаты
    print("\n" + "=" * 80)
    print("РЕЗУЛЬТАТЫ ABLATION STUDY")
    print("=" * 80)

    results = {
        'without_synthetic': {
            'final_accuracy': history1['val_acc'][-1],
            'final_balanced_acc': history1['val_balanced_acc'][-1],
            'best_accuracy': max(history1['val_acc']),
            'best_balanced_acc': max(history1['val_balanced_acc']),
            'avg_train_loss': np.mean(history1['train_loss'][-10:])
        },
        'with_synthetic': {
            'final_accuracy': history2['val_acc'][-1],
            'final_balanced_acc': history2['val_balanced_acc'][-1],
            'best_accuracy': max(history2['val_acc']),
            'best_balanced_acc': max(history2['val_balanced_acc']),
            'avg_train_loss': np.mean(history2['train_loss'][-10:])
        }
    }

    # Печатаем таблицу
    print("\n{:<30} {:<20} {:<20}".format("Метрика", "Без синтетики", "С синтетикой"))
    print("-" * 70)

    for metric in ['final_accuracy', 'final_balanced_acc', 'best_accuracy', 'best_balanced_acc']:
        without = results['without_synthetic'][metric]
        with_syn = results['with_synthetic'][metric]
        improvement = with_syn - without
        print(f"{metric:<30} {without:<20.2f} {with_syn:<20.2f} (+{improvement:.2f})")

    print("\nТренировочные потери (среднее за последние 10 эпох):")
    print(f"  Без синтетики: {results['without_synthetic']['avg_train_loss']:.4f}")
    print(f"  С синтетикой:  {results['with_synthetic']['avg_train_loss']:.4f}")

    # Сохраняем результаты
    with open(config.RESULTS_DIR / "ablation_results.json", 'w') as f:
        json.dump(results, f, indent=2)

    # Строим графики
    plot_training_curves(history1, history2)

    return results


def plot_training_curves(history1, history2):
    """Строим графики обучения"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history1['val_acc'], label='Without Synthetic', marker='o', markersize=3, alpha=0.7)
    axes[0].plot(history2['val_acc'], label='With Synthetic', marker='s', markersize=3, alpha=0.7)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Validation Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Balanced Accuracy
    axes[1].plot(history1['val_balanced_acc'], label='Without Synthetic', marker='o', markersize=3, alpha=0.7)
    axes[1].plot(history2['val_balanced_acc'], label='With Synthetic', marker='s', markersize=3, alpha=0.7)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Balanced Accuracy (%)')
    axes[1].set_title('Validation Balanced Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(config.RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    print(f"\n[Plots] Графики сохранены в {config.RESULTS_DIR / 'training_curves.png'}")
    plt.close()


if __name__ == "__main__":
    # Проверяем GPU
    print(f"[INFO] CUDA доступна: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
        print(f"[INFO] Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB")

    # Запускаем пайплайн
    results = run_complete_pipeline()

    print("\n[COMPLETE] Задание выполнено!")
    print(f"[INFO] Результаты сохранены в {config.RESULTS_DIR}")

[INFO] CUDA доступна: True
[INFO] GPU: NVIDIA A100 80GB PCIe
[INFO] Память: 85GB
ЗАДАНИЕ 2.5: СИНТЕТИЧЕСКИЕ ДАННЫЕ ЧЕРЕЗ STABLE DIFFUSION + CONTROLNET

[1] Загружаем датасеты...

[2] Генерируем синтетические изображения...
[Gen] Загружаем Stable Diffusion для класса 'cat'...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

[Gen] Генерируем 50 изображений для класса 'cat'...


Generating cat:   0%|                                                                                                                                                    | 0/50 [00:00<?, ?it/s]

  0%|          | 0/52 [00:00<?, ?it/s]

...

Evaluating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [00:08<00:00, 18.00it/s]


[Epoch 80/100] Loss: 0.1191 | Acc: 81.32% | Balanced Acc: 81.32%


Evaluating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [00:08<00:00, 17.93it/s]


[Epoch 90/100] Loss: 0.0920 | Acc: 81.75% | Balanced Acc: 81.75%


Training:  56%|███████████████████████████████████████████████████████████████████████████████▋                                                               | 396/711 [00:14<00:06, 52.11it/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



# ACCURACY PLOTS FOR EACH CLASS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch
from pathlib import Path
import json

def plot_per_class_accuracy(history1, history2, test_loader, model1, model2, config, save_dir):
    """
    Строит графики accuracy по каждому классу для двух моделей
    
    Args:
        history1: история обучения модели без синтетики
        history2: история обучения модели с синтетикой  
        test_loader: валидационный датасет
        model1: обученная модель без синтетики
        model2: обученная модель с синтетикой
        config: конфигурация
        save_dir: папка для сохранения графиков
    """
    
    # ✅ Функция для вычисления per-class accuracy
    def compute_per_class_accuracy(model, test_loader, device):
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc="Computing per-class acc"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        # Вычисляем accuracy для каждого класса
        class_accuracy = {}
        class_names = ['airplane', 'car', 'bird', 'cat', 'deer', 
                      'dog', 'frog', 'horse', 'ship', 'truck']
        
        for class_id in range(config.NUM_CLASSES):
            class_mask = np.array(all_labels) == class_id
            if np.sum(class_mask) > 0:
                acc = np.mean(np.array(all_preds)[class_mask] == class_id)
                class_accuracy[class_names[class_id]] = acc * 100
            else:
                class_accuracy[class_names[class_id]] = 0.0
        
        return class_accuracy, all_preds, all_labels
    
    print("[Plots] Вычисляем per-class accuracy...")
    
    # ✅ Вычисляем метрики для обеих моделей
    acc1, preds1, labels1 = compute_per_class_accuracy(model1, test_loader, config.DEVICE)
    acc2, preds2, labels2 = compute_per_class_accuracy(model2, test_loader, config.DEVICE)
    
    class_names = list(acc1.keys())
    acc1_values = list(acc1.values())
    acc2_values = list(acc2.values())
    
    # ✅ ГРАФИК 1: Сравнение accuracy по классам (столбцы)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Per-Class Accuracy Comparison\nSynthetic Data Impact', fontsize=16, fontweight='bold')
    
    # График 1: Столбчатая диаграмма
    x = np.arange(len(class_names))
    width = 0.35
    
    axes[0,0].bar(x - width/2, acc1_values, width, label='Without Synthetic', 
                  alpha=0.8, color='skyblue', edgecolor='navy')
    axes[0,0].bar(x + width/2, acc2_values, width, label='With Synthetic', 
                  alpha=0.8, color='lightcoral', edgecolor='darkred')
    
    axes[0,0].set_xlabel('Classes')
    axes[0,0].set_ylabel('Accuracy (%)')
    axes[0,0].set_title('Per-Class Accuracy (Bar Chart)')
    axes[0,0].set_xticks(x)
    axes[0,0].set_xticklabels(class_names, rotation=45, ha='right')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # ✅ Выделяем редкий класс (cat)
    cat_idx = class_names.index('cat')
    axes[0,0].bar(cat_idx - width/2, acc1_values[cat_idx], width, 
                  alpha=1.0, color='blue', edgecolor='darkblue', linewidth=3, label='Cat (Rare)')
    axes[0,0].bar(cat_idx + width/2, acc2_values[cat_idx], width, 
                  alpha=1.0, color='red', edgecolor='darkred', linewidth=3)
    
    # График 2: Линейный график
    axes[0,1].plot(class_names, acc1_values, 'o-', linewidth=2.5, markersize=8, 
                   label='Without Synthetic', color='blue', alpha=0.8)
    axes[0,1].plot(class_names, acc2_values, 's-', linewidth=2.5, markersize=8, 
                   label='With Synthetic', color='red', alpha=0.8)
    axes[0,1].set_xlabel('Classes')
    axes[0,1].set_ylabel('Accuracy (%)')
    axes[0,1].set_title('Per-Class Accuracy (Line Chart)')
    axes[0,1].tick_params(axis='x', rotation=45)
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # ✅ График 3: Разница в accuracy (Improvement)
    improvements = np.array(acc2_values) - np.array(acc1_values)
    colors = ['green' if imp > 0 else 'red' for imp in improvements]
    
    bars = axes[1,0].bar(class_names, improvements, alpha=0.8, color=colors, edgecolor='black')
    axes[1,0].axhline(y=0, color='black', linestyle='-', linewidth=1)
    axes[1,0].set_xlabel('Classes')
    axes[1,0].set_ylabel('Improvement (%)')
    axes[1,0].set_title('Accuracy Improvement (With Synthetic - Without)')
    axes[1,0].tick_params(axis='x', rotation=45)
    axes[1,0].grid(True, alpha=0.3)
    
    # Добавляем значения на столбцы
    for bar, imp in zip(bars, improvements):
        height = bar.get_height()
        axes[1,0].text(bar.get_x() + bar.get_width()/2., height + (0.5 if height > 0 else -0.8),
                      f'{imp:+.1f}%', ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')
    
    # ✅ График 4: Heatmap матрицы ошибок (только для cat и других)
    cm1 = confusion_matrix(labels1, preds1, normalize='true')
    cm2 = confusion_matrix(labels2, preds2, normalize='true')
    
    cat_cm1 = cm1[3, :] * 100  # Строка cat (класс 3)
    cat_cm2 = cm2[3, :] * 100
    
    x_labels = [f'Pred: {name[:3]}' for name in class_names]
    
    sns.heatmap([cat_cm1, cat_cm2], annot=True, fmt='.1f', cmap='RdYlBu_r', 
                xticklabels=x_labels, yticklabels=['Real Cat (No Synth)', 'Real Cat (w/ Synth)'],
                ax=axes[1,1], cbar_kws={'label': 'Prediction Probability (%)'})
    axes[1,1].set_title('Cat Predictions Distribution\n(Rows: True Cat, Columns: Predicted Class)')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'per_class_accuracy_comparison.png', dpi=300, bbox_inches='tight')
    plt.savefig(save_dir / 'per_class_accuracy_comparison.pdf', bbox_inches='tight')
    plt.close()
    
    print(f"✅ Графики per-class accuracy сохранены: {save_dir / 'per_class_accuracy_comparison.png'}")
    
    # ✅ Таблица результатов
    print("\n" + "="*80)
    print("PER-CLASS ACCURACY COMPARISON")
    print("="*80)
    print(f"{'Class':<12} {'No Synth':<10} {'With Synth':<12} {'Δ':<8}")
    print("-"*80)
    
    total_improvement = 0
    for i, class_name in enumerate(class_names):
        no_synth = acc1_values[i]
        with_synth = acc2_values[i]
        delta = with_synth - no_synth
        total_improvement += delta
        print(f"{class_name:<12} {no_synth:>7.1f}% {with_synth:>11.1f}% {delta:>+6.1f}%")
    
    avg_improvement = total_improvement / len(class_names)
    print("-"*80)
    print(f"AVERAGE:       {'':<10} {'':<12} {avg_improvement:+6.1f}%")
    
    # ✅ Сохраняем данные в JSON
    results = {
        'without_synthetic': acc1,
        'with_synthetic': acc2,
        'improvements': {name: acc2[name] - acc1[name] for name in class_names},
        'best_improved_class': max(acc2.items(), key=lambda x: x[1] - acc1.get(x[0], 0))[0],
        'worst_changed_class': min(acc2.items(), key=lambda x: x[1] - acc1.get(x[0], 0))[0]
    }
    
    with open(save_dir / 'per_class_accuracy.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"✅ Детальные результаты: {save_dir / 'per_class_accuracy.json'}")
    
    return results


print("\n[5] Анализ accuracy по классам...")

# Строим графики
results = plot_per_class_accuracy(
    history1, history2, 
    data_loaders['test'], 
    model1, model2, 
    config, 
    config.RESULTS_DIR
)